In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings(action='ignore')
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [3]:
df = pd.read_csv(r"C:\MY FILES\EMAIL SPAM DETECTION Project\email.csv")


In [4]:
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...
5571,ham,Rofl. Its true to its name


In [5]:
df['Category'].value_counts()

Category
ham               4825
spam               747
{"mode":"full"       1
Name: count, dtype: int64

In [6]:
df.iloc[5572]

Category     {"mode":"full"
Message     isActive:false}
Name: 5572, dtype: object

In [7]:
df = df.drop(index=5572)
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [8]:
le = LabelEncoder()
df['Category'] = le.fit_transform(df['Category'])  # 1: spam, 0: ham

In [9]:
df

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will ü b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [10]:
df.iloc[0]

Category                                                    0
Message     Go until jurong point, crazy.. Available only ...
Name: 0, dtype: object

In [11]:
df.iloc[2]

Category                                                    1
Message     Free entry in 2 a wkly comp to win FA Cup fina...
Name: 2, dtype: object

In [12]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [14]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [15]:
ps = WordNetLemmatizer()
stopwords_set = set(stopwords.words('english'))

In [17]:
def clean_row(row):
    row = row.lower()
    row = re.sub('[^a-zA-Z]', ' ', row)
    tokens = row.split()
    email = [ps.lemmatize(word) for word in tokens if word not in stopwords_set]
    return ' '.join(email)

In [18]:
df['Message']

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                 Will ü b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: Message, Length: 5572, dtype: object

In [19]:
df['Message'] = df['Message'].apply(clean_row)

In [20]:
df['Message']

0       go jurong point crazy available bugis n great ...
1                                 ok lar joking wif u oni
2       free entry wkly comp win fa cup final tkts st ...
3                     u dun say early hor u c already say
4                     nah think go usf life around though
                              ...                        
5567    nd time tried contact u u pound prize claim ea...
5568                            b going esplanade fr home
5569                                 pity mood suggestion
5570    guy bitching acted like interested buying some...
5571                                       rofl true name
Name: Message, Length: 5572, dtype: object

In [21]:
vectorizer = TfidfVectorizer(max_features=9000, lowercase=False, ngram_range=(1, 2))
X = df['Message']
Y = df['Category']

In [22]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


In [23]:
vec_train_data = vectorizer.fit_transform(X_train).toarray()
vec_test_data = vectorizer.transform(X_test).toarray()

In [24]:
nb_model = MultinomialNB()
nb_model.fit(vec_train_data, Y_train)

MultinomialNB()

In [25]:
nb_pred = nb_model.predict(vec_test_data)

print("Classification Report:")
print(classification_report(Y_test, nb_pred, target_names=['Ham', 'Spam']))

Classification Report:
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



In [26]:
# Test on a sample email
sample_email = "Congratulations! You've won a $1000 Walmart gift card. Click here to claim your prize."
cleaned_sample_email = clean_row(sample_email)
vec_sample_email = vectorizer.transform([cleaned_sample_email]).toarray()
sample_prediction = nb_model.predict(vec_sample_email)

In [27]:
print(f"\nSample Email: {sample_email}")
print(f"Prediction: {'Spam' if sample_prediction[0] == 1 else 'Not Spam'}")


Sample Email: Congratulations! You've won a $1000 Walmart gift card. Click here to claim your prize.
Prediction: Spam


In [28]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(kernel='linear', random_state=42),
    "Gaussian NB": GaussianNB(),
    "KNN": KNeighborsClassifier()
}

In [29]:
results = {}

for model_name, model in models.items():
    model.fit(vec_train_data, Y_train)
    predictions = model.predict(vec_test_data)
    
    accuracy = accuracy_score(Y_test, predictions)
    results[model_name] = {
        "accuracy": accuracy,
        "classification_report": classification_report(Y_test, predictions, target_names=['Ham', 'Spam'], output_dict=True)
    }
    print(f"--- {model_name} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(Y_test, predictions, target_names=['Ham', 'Spam']))
    print("\n")

--- Naive Bayes ---
Accuracy: 0.9713
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       1.00      0.79      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115



--- Decision Tree ---
Accuracy: 0.9695
              precision    recall  f1-score   support

         Ham       0.97      0.99      0.98       966
        Spam       0.94      0.83      0.88       149

    accuracy                           0.97      1115
   macro avg       0.96      0.91      0.93      1115
weighted avg       0.97      0.97      0.97      1115



--- Random Forest ---
Accuracy: 0.9749
              precision    recall  f1-score   support

         Ham       0.97      1.00      0.99       966
        Spam       1.00      0.81      0.90       149

    accuracy                           0.97      1115
   macro 

In [30]:
print("Model Performance Comparison:")
for model_name, metrics in results.items():
    print(f"{model_name}: Accuracy = {metrics['accuracy']:.4f}")

Model Performance Comparison:
Naive Bayes: Accuracy = 0.9713
Decision Tree: Accuracy = 0.9695
Random Forest: Accuracy = 0.9749
SVM: Accuracy = 0.9883
Gaussian NB: Accuracy = 0.8933
KNN: Accuracy = 0.9175


In [33]:
sample_email = "Hi Team, I hope this email finds you well. I wanted to inform you that our weekly team meeting has been rescheduled to  Thursday at 2 PM due to scheduling conflicts. Please let me know if this time works for everyone. Best"
cleaned_sample_email = clean_row(sample_email)
vec_sample_email = vectorizer.transform([cleaned_sample_email]).toarray()

print("\nSample Email Classification:")
for model_name, model in models.items():
    if model_name == "Gaussian NB":
        prediction = model.predict(vec_sample_email)
    else:
        prediction = model.predict(vec_sample_email)
    print(f"{model_name}: {'Spam' if prediction[0] == 1 else 'Ham'}")


Sample Email Classification:
Naive Bayes: Ham
Decision Tree: Ham
Random Forest: Ham
SVM: Ham
Gaussian NB: Ham
KNN: Ham


In [32]:

sample_email = '''Hi Team,
I hope this email finds you well. I wanted to inform you that our weekly team meeting has been rescheduled to 
Thursday at 2 PM due to scheduling conflicts. Please let me know if this time works for everyone.
Best regards'''

cleaned_sample_email = clean_row(sample_email)
vec_sample_email = vectorizer.transform([cleaned_sample_email]).toarray()

print("\nSample Email Classification:")
for model_name, model in models.items():
    if model_name == "Gaussian NB":
        prediction = model.predict(vec_sample_email)
    else:
        prediction = model.predict(vec_sample_email)
    print(f"{model_name}: {'Spam' if prediction[0] == 1 else 'Ham'}")


Sample Email Classification:
Naive Bayes: Ham
Decision Tree: Ham
Random Forest: Ham
SVM: Ham
Gaussian NB: Ham
KNN: Ham
